# 📗 지식 그래프: 트리플과 온톨로지 설계

지난 시간에는 의학 논문에서 <strong>개체(엔티티)</strong>를 뽑았습니다. clopidogrel·CYP2C19·myopathy 같은 이름들이 목록으로 나왔죠. 그런데 "**clopidogrel 이 어느 유전자와 얽혀 있지?**"라고 물으면 개체 목록으로는 답할 수 없습니다. 이름만 있고 **개체 사이의 연결**이 없기 때문입니다.

이번 시간에는 그 연결을 담는 그릇, <strong>트리플(주어-관계-목적어)</strong>을 배웁니다. 논문 문장을 트리플로 쪼개고, 트리플을 그래프(노드-관계-노드)로 옮기고, **노드를 무엇으로 식별할지** 정하고, 아무 관계나 허용하지 않도록 **관계 시그니처**로 스키마를 설계합니다. 실제 추출과 그래프 적재는 다음 시간에 이어집니다.

## ⏪ 복습: 지난 시간까지

- **개체 추출(NER)**: 논문 본문에서 개체를 뽑아 `{name, type}` 형태로 정리했습니다.
- **구조화 출력**: 모델 답을 자유 문장이 아니라 **정해진 구조**로 받았습니다. 트리플을 담을 때 이 스킬을 다시 씁니다.
- **지식그래프 적재**: 노드는 `MERGE (c:Compound {id: 'Compound::DB00381'})` 처럼 **id 로** 만들고 이름은 속성으로 붙였습니다. **오늘 2-1 에서** 그 규칙이 왜 그랬는지 데이터로 다시 확인합니다.

> **데이터 출처**
>
> | 데이터 | 원본 | 이용 조건 |
> |---|---|---|
> | 논문 본문 발췌 (`pgx_*.jsonl`) | PubMed Central Open Access Subset (pmcid 를 각 문서에 적어 두었습니다) | CC BY |
> | 이름 -> id 사전 (`name2id.json`) | Hetionet v1.0 (https://het.io) + RxNav(NLM) 약물 동의어 | CC0 / 공개 |
> | 큐레이션 관계 (`hetionet_curated.jsonl`) | Hetionet v1.0 에서 CC0 출처만 골라낸 부분 | CC0 |
>
> 논문에서 뽑은 관계는 **그 논문이 그렇게 보고했다**는 뜻이고, Hetionet 관계는 **2016년에 정리된** 문헌 근거라는 뜻입니다. 둘 다 "효능이 입증됐다"는 말이 아닙니다. 이 구분을 트리플 속성 `evidence_level` 로 남기는 법을 오늘 배웁니다.

**오늘의 목표**

**1. 논문 문장을 트리플로**
- [ ] 논문 문장 하나를 트리플 여러 개로 쪼개고, 그래프로 옮기려면 **타입 두 칸**이 더 필요한 이유를 안다.

**2. 온톨로지 설계: 무엇을 허용할 것인가**
- [ ] (2-1) 노드의 **키를 이름이 아니라 id 로** 두어야 하는 이유를 데이터로 확인하고, 사전에 없는 이름을 **`:Candidate`** 로 격리하는 규칙을 세운다.
- [ ] (2-2) **관계 시그니처**(주어 타입·목적어 타입·판정 기준)로 허용 관계 8종을 설계하고, **온톨로지가 구분하지 않기로 한 것은 되찾을 수 없다**는 것을 `BINDS` 로 확인한다.

아래 준비 셀 두 개를 먼저 실행하세요. 이 노트북은 모델도 Neo4j 도 쓰지 않지만, 모든 실습 노트북이 같은 준비 셀로 시작하고 두 번째 셀의 사전을 뒤에서 씁니다.

In [ ]:
# [제공 코드] OpenAI 키 준비 - 이 셀은 실행만 하세요.
# 14~16일차와 같은 방식입니다: .env 파일에 넣어 둔 OPENAI_API_KEY 를 읽어 옵니다.
import os

from dotenv import load_dotenv

load_dotenv(".env")       # 같은 폴더의 .env
load_dotenv("../.env")    # 정답 폴더에서 실행하는 경우

# 키를 먼저 확인합니다. 모델을 만든 뒤에 검사하면 인증 오류가 먼저 나서 이 안내가 묻힙니다.
if not os.getenv("OPENAI_API_KEY"):
    raise RuntimeError(
        "이 노트북은 실제 OpenAI 호출이 필요합니다 - OPENAI_API_KEY 를 찾지 못했습니다.\n"
        "  1) 일차 폴더에서  cp .env.example .env\n"
        "  2) .env 를 열어 본인 키를 채우세요\n"
        "  3) 커널을 재시작한 뒤 이 셀부터 다시 실행하세요")

from langchain_openai import ChatOpenAI

MODEL_NAME = "gpt-5.6-luna"


def make_model():
    """이 단원이 쓰는 챗 모델.

    이 모델은 temperature 를 받지 않습니다(0 을 넘기면 400 이 옵니다). 그래서 출력을 고정할
    손잡이가 없고, 같은 입력에도 답이 흔들립니다(자동화 파이프라인 단원 6절이 그 흔들림을
    여러 번 돌려 잽니다).
    """
    return ChatOpenAI(model=MODEL_NAME)


print("모델:", MODEL_NAME)

In [ ]:
# [제공 코드] 이름 -> id 사전 준비: 이 셀은 실행만 하세요.
# 사전을 어떻게 만드는지는 엔티티 정규화 단원에서 배웁니다. 여기서는 읽어 쓰기만 합니다.
import json
import re
from pathlib import Path

# 파일의 최상위 키는 여덟 개인데 이 노트북이 읽는 것은 셋이다
#   entries: 정규화한 이름 -> {id, label} / genes: 유전자 기호 -> id / ambiguous: 한 이름이 두 타입에 걸린 경우
NAME2ID = json.loads(Path("data/name2id.json").read_text(encoding="utf-8"))


# 표기 차이를 눌러 사전 조회용 키로 만든다. lookup_id 함수가 호출한다
def normalize_name(name):
    """이름 매칭용 정규화: 소문자, 앞뒤 공백 제거, 연속 공백 한 칸, 끝의 괄호 주석 제거."""
    s = re.sub(r"\s+", " ", name).strip().lower()
    return re.sub(r"\s*\([^)]*\)$", "", s).strip()


# 이름을 노드 id 로 바꿔 돌려준다. 트리플을 그래프에 적재할 때 쓴다
def lookup_id(name, node_type):
    """이름과 타입으로 Hetionet id 를 찾아 (id, 사유) 두 칸으로 돌려준다.

    사유는 셋 중 하나다. 못 붙은 이유가 둘로 갈리므로 id 만으로는 구별할 수 없다.
      "hit"        붙었다. 첫 칸이 그 id 다
      "miss"       사전에 그 이름이 없다(또는 타입이 다르다). 첫 칸은 None
      "ambiguous"  후보가 둘 이상이라 이름만으로는 못 고른다. 첫 칸은 None
    """
    if node_type == "Gene":
        # 유전자 기호는 대소문자가 곧 뜻이다. 소문자로 누르면 CAT·SET 같은 흔한 단어가 유전자로 잡힌다
        found = NAME2ID["genes"].get(name.strip())
        return (found, "hit") if found else (None, "miss")
    key = normalize_name(name)   # 유전자 말고는 표기 차이를 눌러 놓고 찾는다
    if key in NAME2ID["ambiguous"]:
        return None, "ambiguous"      # 한 이름이 서로 다른 타입 두 곳에 걸린 경우
    entry = NAME2ID["entries"].get(key)
    # 타입까지 맞아야 같은 개체다. obesity 는 Disease 이면서 Symptom 이라 타입을 안 보면 엉뚱하게 붙는다
    if entry and entry["label"] == node_type:
        return entry["id"], "hit"
    return None, "miss"


print("사전 항목:", len(NAME2ID["entries"]),
      "/ 유전자 기호:", len(NAME2ID["genes"]),
      "/ 애매한 이름:", len(NAME2ID["ambiguous"]))

---
# 1. 논문 문장을 트리플로

**트리플**(주어-관계-목적어)도, 그것을 **노드-관계-노드**로 옮기는 **LPG** 도, `MERGE` 가 여러 번 넣어도 불어나지 않는다는 것도 그래프DB 단원과 지식그래프 적재 단원에서 이미 배웠습니다. 여기서는 그 셋을 **논문 문장**에 적용해 봅니다. 오늘 새로운 것은 둘입니다.

- 논문 한 문장에는 **사실이 여럿 뭉쳐 있어** 트리플 여러 개로 쪼개진다는 것
- 그래프로 옮기려면 세 칸으로는 부족하고 **타입 두 칸이 더** 필요하다는 것

이 둘을 확인하고 나면 질문이 하나 남습니다. **노드를 무엇으로 알아볼 것인가.** 2절이 그 답을 정합니다.

In [ ]:
# 논문 문장 하나를 사실 단위로 손으로 쪼갠다. 트리플은 (주어, 관계, 목적어) 튜플로 적는다
sentence = 'Tapinarof is the first-in-class nonsteroidal, topical AHR agonist that was approved by the FDA in 2022 for the treatment of plaque psoriasis and in 2024 for the treatment of atopic dermatitis.'
print(sentence)

In [ ]:
# 이 한 문장에 사실이 셋 들어 있다. 주어는 셋 다 Tapinarof 이고 관계와 목적어가 달라진다
triples = [
    ("Tapinarof", "결합한다", "AHR"),
    ("Tapinarof", "치료한다", "plaque psoriasis"),
    ("Tapinarof", "치료한다", "atopic dermatitis"),
]

for subj, rel, obj in triples:
    print(f"({subj}, {rel}, {obj})")

> 한 문장에서 사실이 **세 개** 나왔습니다. 세 트리플 모두 주어가 **Tapinarof** 입니다. 논문에서 그 문단의 주인공이 되는 개체를 주어로 모으면 사실이 한곳에 쌓입니다.
>
> 관계 이름을 지금은 한국어("치료한다")로 적었습니다. 실제 그래프에 넣을 이름은 **2-2 에서 정해진 목록** 중에서 고르게 됩니다.

### 그래프로 옮길 때는 칸이 둘 더 필요합니다
세 칸만 보면 코드는 노드에 `:Compound` 를 붙일지 `:Disease` 를 붙일지 알 수 없습니다. 그래서 아래 코드부터는 트리플을 `(주어, 주어 타입, 관계, 목적어, 목적어 타입)` **다섯 칸**으로 넘깁니다. 사실 자체는 여전히 세 칸이고, 늘어난 두 칸은 그 사실을 그래프로 옮기기 위한 **표지**입니다.

노드를 무엇으로 알아볼지는 아직 정하지 않았습니다. 우선 가장 소박한 답인 **이름**으로 적어 보고, 2절에서 그 방식이 왜 안 되는지 봅니다.

In [ ]:
# 첫 시도: 이름을 키로 두고 MERGE 문을 만든다. 다음 절에서 이 방식의 문제를 본다
def merge_by_name(triple):
    """다섯 칸 트리플을 MERGE 세 줄짜리 Cypher 문자열로 바꾼다(노드 키는 이름)."""
    subj, subj_type, rel, obj, obj_type = triple   # 다섯 칸: 주어, 주어 타입, 관계, 목적어, 목적어 타입
    # 아래 세 줄이 트리플의 세 칸과 짝이다: 주어 노드, 목적어 노드, 둘을 잇는 관계
    # CREATE 가 아니라 MERGE 라서 같은 트리플을 여러 번 넣어도 노드·관계가 불어나지 않는다
    return (f"MERGE (a:{subj_type} {{name: '{subj}'}})\n"
            f"MERGE (b:{obj_type} {{name: '{obj}'}})\n"
            f"MERGE (a)-[:{rel}]->(b)")

cy = merge_by_name(("Tapinarof", "Compound", "TREATS", "plaque psoriasis", "Disease"))
print(cy)

### ✅ 바로 확인 퀴즈

**1.** "Simvastatin 을 복용한 49명 중 12명이 SLCO1B1 활성 저하 표현형을 가지고 있었다"에서 **개체 사이의 연결**로 뽑을 수 있는 트리플은 몇 개일까요?

<details><summary>정답 보기</summary>

**한 개**입니다. Simvastatin 과 SLCO1B1 이 얽혀 있다는 사실 하나뿐이고, "49명 중 12명"은 개체가 아니라 그 사실의 **크기**입니다. 숫자를 개체로 만들면 쓸모없는 노드가 그래프에 쌓입니다.

</details>

**2.** 사실 하나는 세 칸인데 `merge_by_name` 은 왜 다섯 칸을 받을까요? 늘어난 두 칸은 사실의 일부일까요?

<details><summary>정답 보기</summary>

늘어난 두 칸은 **주어 타입과 목적어 타입**입니다. 사실의 일부가 아니라, 그 사실을 그래프로 옮길 때 노드에 어떤 레이블을 붙일지 알려 주는 **표지**입니다. 이름만으로는 `plaque psoriasis` 가 `:Disease` 인지 `:Symptom` 인지 코드가 정할 수 없습니다.

</details>

---
# 2. 온톨로지 설계: 무엇을 허용할 것인가

**1. 트리플**에서는 사실 하나를 **적는 법**을 배웠습니다. 여기서는 그 사실을 **무엇으로 식별하고 무엇까지 허용할지**를 정합니다. 뽑기 전에 정해 두어야 하는 것들입니다.

- **2-1** 노드의 키를 이름이 아니라 id 로 두고, 사전에 없는 이름은 `:Candidate` 로 격리합니다.
- **2-2** 관계마다 (주어 타입, 목적어 타입, 판정 기준)을 못 박습니다.

## 2-1. 노드의 키: 이름이 아니라 id

앞 절에서 `MERGE (a:Compound {name: 'Tapinarof'})` 라고 썼습니다. 이름이 **유일한 키**라는 가정이 깔려 있는데, 의학 이름은 그렇지 않습니다. 이 가정이 깨지면 노드가 **개체**가 아니라 **적힌 글자**를 가리키게 됩니다. 글자가 같아도 다른 개체일 수 있고, 같은 개체라도 논문마다 다르게 적습니다. 의료 그래프에서는 심각한 오류입니다.

### 실제 데이터로 확인하기
준비 셀에서 읽은 사전(`NAME2ID`)에는 **`ambiguous`** 칸이 있습니다. 이름 하나가 **종류가 다른 노드 둘**에 걸린 경우를 모아 둔 것입니다.

In [ ]:
print("애매한 이름:", len(NAME2ID["ambiguous"]), "건")

In [ ]:
for name, hits in NAME2ID["ambiguous"].items():
    # 같은 글자인데 id 도 타입도 다르다. 이름만 보고는 어느 쪽을 말하는지 정할 수 없다
    kinds = ", ".join(f"{h['label']}({h['id']})" for h in hits)
    print(f"  {name:<24} {kinds}")

### 문법: 이름을 id 로 바꾸는 조회
준비 셀의 `lookup_id(name, node_type)` 가 그 일을 합니다. 붙었는지만이 아니라 **못 붙었으면 왜 못 붙었는지**까지 알아야 하므로, 값 하나가 아니라 `(id, 사유)` **두 칸**을 돌려줍니다.

| 반환값 | 뜻 | 어떻게 할 것인가 |
|---|---|---|
| `("Compound::DB00758", "hit")` | 사전에 있고 타입도 맞는다 | 그 id 로 `MERGE` |
| `(None, "miss")` | 사전이 모르는 이름 | **`:Candidate`** 로 격리 |
| `(None, "ambiguous")` | 후보가 둘 이상이라 이름만으로는 못 고른다 | 오늘은 **`:Candidate`** 로 격리 |

둘 다 `None` 이지만 **뜻이 다릅니다.** `miss` 는 사전이 그 이름을 아예 모르는 것이고, `ambiguous` 는 사전이 후보를 여럿 아는데 이름만으로는 어느 쪽인지 못 고르는 것입니다. 첫 칸만 보면 이 둘이 같아 보여서, 뒤에서 후보를 다르게 다루고 싶어도 가를 수가 없습니다. 그래서 사유를 함께 돌려줍니다.

**사전에 없다고 틀린 이름은 아닙니다.** 사전이 2016년에 정리된 것이라 그 뒤에 나온 약은 없습니다. 그래서 버리지 않고 `:Candidate` 로 따로 세워 둡니다. 다음 단원들이 이 후보를 정규화하고, 붙일 곳이 생기면 그때 승격합니다.

**`ambiguous` 는 영영 못 고르는 것이 아닙니다.** 이름 한 낱말만 보면 못 고르지만, 그 이름이 어떤 문장에서 나왔고 그래프에서 무엇과 이어져 있는지를 보면 고를 수 있습니다. 문장과 후보 목록을 함께 모델에게 주고 고르게 할 수도 있습니다. 다만 그러려면 이 조회 함수가 가진 것보다 많은 정보가 필요합니다. 오늘은 후보로 남겨 두고, **엔티티 정규화 단원**에서 문맥으로 고르는 법을 배웁니다.

In [ ]:
# 논문에서 뽑은 이름 네 개를 사전에 물어본다. 결과가 셋으로 갈린다
# id 가 둘 다 None 이어도 사유가 다르다. 그 사유가 뒤에서 후보를 어떻게 다룰지를 가른다
for name, node_type in [("Clopidogrel", "Compound"), ("CYP2C19", "Gene"),
                        ("Tapinarof", "Compound"), ("obesity", "Disease")]:
    found, why = lookup_id(name, node_type)   # 두 칸을 한 줄에 풀어 받는다
    print(f"{name:<14} {node_type:<10} -> {found} ({why})")

> `Clopidogrel`·`CYP2C19` 는 id 가 나왔습니다(`hit`). `Tapinarof` 는 **2022년에 승인된 약**이라 2016년 사전에 없어 `miss` 이고, `obesity` 는 후보가 둘이라 `ambiguous` 입니다. id 를 못 받은 이 두 이름이 `:Candidate` 로 격리할 대상입니다. 오늘 하는 일은 둘 다 격리로 같지만, 사유를 남겨 두었으므로 뒤 단원에서 `ambiguous` 만 따로 골라 문맥으로 붙일 수 있습니다.

### 문법: 확정판 매핑 함수
이제 키가 정해졌으니 트리플을 Cypher 로 옮기는 함수를 다시 씁니다. `merge_by_name` 과 다른 점은 **중괄호 안이 `name` 에서 `id` 로 바뀐 것 하나**입니다.

<img src="images/triple_to_lpg.png" width="740">

In [ ]:
# 확정판: 노드의 키는 id 다. 이름은 뒤에서 속성으로 붙인다(다음 시간)
def triple_to_cypher(triple):
    """다섯 칸 트리플을 MERGE 세 줄짜리 Cypher 문자열로 바꾼다(노드 키는 id)."""
    subj_id, subj_type, rel, obj_id, obj_type = triple   # 이름 자리에 이미 해소된 id 가 들어온다
    # id 는 사전이 준 값이라 따옴표가 들어 있지 않다. 그래서 문장에 그대로 끼워 넣어도 안전하다
    return (f"MERGE (a:{subj_type} {{id: '{subj_id}'}})\n"
            f"MERGE (b:{obj_type} {{id: '{obj_id}'}})\n"
            f"MERGE (a)-[:{rel}]->(b)")

print(triple_to_cypher(("Compound::DB00758", "Compound", "BINDS", "Gene::1557", "Gene")))

### 잠깐: 따라하기에 쓸 데이터

아래 따라하기부터는 **단위 프로젝트에서 실제로 쓸 데이터**로 연습합니다. 교안 데모는 의학 논문이고 따라하기는 **파이썬 라이브러리 문서**입니다. 배운 기술이 도메인을 갈아 끼워도 그대로 서는지 확인하려는 것입니다.

| 무엇 | 어디서 왔나 |
|---|---|
| 대상 라이브러리 다섯 | pandas · matplotlib · seaborn · langchain · langgraph. 이 과정에서 실제로 배운 것들입니다 |
| 수집물 | 각 저장소의 공식 문서·API 레퍼런스·릴리스 노트를 GitHub 에서 내려받은 것 |
| 이용 조건 | 각 저장소 라이선스. 원본 링크를 함께 보존합니다 |

그중 따라하기에 쓸 만큼만 추려 `data/` 에 세 파일로 두었습니다.

| 파일 | 담긴 것 | 어디에 쓰나 |
|---|---|---|
| `pylibs_api_names.json` | API 개체 이름 사전 | 아래 따라하기: 이름을 id 로 바꿀 때 |
| `pylibs_triples.jsonl` | 문서에서 뽑아 둔 트리플 | 2-2 끝 따라하기: Cypher 로 옮길 때 |
| `pylibs_docs.jsonl` | 문서 본문 발췌 두 편 | 다음 시간: 모델로 직접 뽑을 때 |

**먼저 눈으로 봅니다.** 새 데이터는 무엇이 들어 있는지 보고 나서 쓰는 것이 순서입니다.

In [ ]:
# 이름 사전부터 본다. 최상위 키가 resolved·ambiguous 둘인데, 그 둘이 갈리는 이유가 이 절의 주제다
PYLIB_NAMES = json.loads(Path("data/pylibs_api_names.json").read_text(encoding="utf-8"))
print("이름 사전:", len(PYLIB_NAMES["resolved"]), "개 / 애매한 이름:", len(PYLIB_NAMES["ambiguous"]), "개")

In [ ]:
# 붙는 이름은 '라이브러리::이름' 하나로 끝난다. 어느 라이브러리 것인지가 id 에 담겨 있다
for name, node_id in list(PYLIB_NAMES["resolved"].items())[:3]:
    print(f"  {name:<18} -> {node_id}")

In [ ]:
# 애매한 이름은 후보가 여럿이다. 이름만 보고는 어느 라이브러리 것인지 정할 수 없다
for name, candidates in list(PYLIB_NAMES["ambiguous"].items())[:3]:
    print(f"  {name:<18} -> {candidates}")

In [ ]:
# 트리플 파일도 본다. 칸 이름이 교안의 다섯 칸 트리플과 같아서 그대로 넘길 수 있다
pylib_rows = [json.loads(line) for line
              in Path("data/pylibs_triples.jsonl").read_text(encoding="utf-8").splitlines()]
print("트리플:", len(pylib_rows), "건 / 칸:", list(pylib_rows[0]))

이 그래프의 관계는 다섯 종이고 **두 축**으로 갈립니다. 문서가 어느 축이냐에 따라 만들 수 있는 관계가 정해집니다.

| 축 | 관계 | 뜻 |
|---|---|---|
| 사용 | `DEMONSTRATES` | 문서가 그 API 를 예제로 보여 준다 |
| 사용 | `RAISES` | 그 API 가 이 예외를 낸다 |
| 변경 | `INCLUDES` | 릴리스 문서가 그 변경을 담고 있다 |
| 변경 | `AFFECTS` | 그 변경이 이 API 를 바꾼다 |
| 변경 | `REFERENCES` | 그 변경이 이 이슈를 가리킨다 |

매뉴얼·API 레퍼런스는 **사용 축**, 릴리스·변경이력은 **변경 축**입니다. 매뉴얼에서 `AFFECTS` 가 나오면 그건 잘못 뽑은 것입니다.

In [ ]:
# 다섯 종을 뜻과 함께 세어 본다. 이 두 문서에 없는 관계는 0 으로 나온다
from collections import Counter

PYLIB_MEANING = {
    "DEMONSTRATES": "문서가 그 API 를 예제로 보여 준다",
    "RAISES":       "그 API 가 이 예외를 낸다",
    "INCLUDES":     "릴리스 문서가 그 변경을 담고 있다",
    "AFFECTS":      "그 변경이 이 API 를 바꾼다",
    "REFERENCES":   "그 변경이 이 이슈를 가리킨다",
}
counts = Counter(r["relation"] for r in pylib_rows)
for rel, meaning in PYLIB_MEANING.items():
    print(f"  {rel:<14} {counts.get(rel, 0):>2}건   {meaning}")

In [ ]:
# 한 건을 통째로 펼쳐 본다. evidence 까지 붙어 있어 근거를 되짚을 수 있다
print(json.dumps(pylib_rows[0], ensure_ascii=False, indent=1))

> 관계 이름이 `DEMONSTRATES`·`INCLUDES`·`AFFECTS` 입니다. 의학 그래프의 `TREATS`·`BINDS` 와 겹치는 것이 하나도 없습니다. **도메인이 바뀌면 온톨로지가 통째로 바뀝니다.** 그런데 칸의 **모양**(주어·주어 타입·관계·목적어·목적어 타입·근거)은 그대로입니다. 바뀌지 않는 것이 그것뿐이라, 오늘 배운 코드가 도메인을 갈아 끼워도 그대로 섭니다.

### 🖐️ 함께 따라하기: 라이브러리 문서에서도 같은 문제가 난다

**단위 프로젝트에서 만들 그래프로 연습합니다.** 파이썬 라이브러리(pandas·matplotlib·seaborn·langchain·langgraph) 공식 문서에서 뽑은 개체 이름 사전이 `data/pylibs_api_names.json` 에 있습니다. 의학 사전과 칸 이름만 다르고 구조는 같습니다.

```text
{"resolved": {"쓰인 이름": "라이브러리::이름", ...},
 "ambiguous": {"쓰인 이름": ["라이브러리::이름", ...], ...}}
```

아래 이름 다섯 개를 사전에 물어 세 갈래로 가르세요. 사전의 키 이름을 그대로 씁니다.

| 갈래 | 뜻 | 담을 리스트 |
|---|---|---|
| `resolved` 에 있다 | 후보가 하나뿐이라 **id 가 확정된** 이름 | `resolved_names` |
| `ambiguous` 에 있다 | 후보가 둘 이상이라 **이름만으로는 못 고르는** 이름 | `ambiguous_names` |
| 둘 다 없다 | 사전이 **모르는** 이름 | `missing_names` |

각 리스트에는 이름만 담고, 마지막에 세 리스트의 길이와 내용을 출력합니다.

```text
["rugplot", "DataFrame", "StateGraph", "read_parquet", "to_feather"]
```

**확인 기준**: 확정 2개(`rugplot`·`read_parquet`), 애매 2개(`DataFrame`·`StateGraph`), 사전에 없음 1개(`to_feather`)입니다. `DataFrame` 은 matplotlib 문서에도 pandas 문서에도 나오고 `StateGraph` 는 langchain·langgraph 양쪽에 나와서, 이름만으로는 **어느 라이브러리의 것인지 정할 수 없습니다.** 의학 사전에서 `obesity` 가 그랬던 것과 같은 자리입니다.

In [ ]:
# 🖐️ 함께 따라하기 (아래 순서대로 직접 작성해 보세요)
# 1) data/pylibs_api_names.json 을 읽어 PYLIBS 에 담는다
# 2) 위 다섯 이름을 names 라는 리스트에 담는다
# 3) 이름마다 resolved 에 있으면 resolved_names, ambiguous 에 있으면 ambiguous_names,
#    둘 다 아니면 missing_names 에 담는다
# 4) 세 리스트의 길이와 내용을 출력한다

### ✅ 바로 확인 퀴즈

**1.** `MERGE (n:Disease {name: 'obesity'})` 와 `MERGE (n:Symptom {name: 'obesity'})` 를 차례로 실행하면 노드가 몇 개 생길까요? 그런데도 이름을 키로 쓰면 안 되는 이유는?

<details><summary>정답 보기</summary>

레이블이 달라 노드는 **두 개** 생깁니다. 문제는 그다음입니다. 추출기가 같은 이름을 어떤 때는 `Disease` 로, 어떤 때는 `Symptom` 으로 붙이면 **같은 사실이 두 노드에 흩어지고**, 반대로 두 논문이 서로 다른 개체를 같은 이름으로 부르면 **다른 사실이 한 노드에 뭉칩니다.** id 를 쓰면 이름 표기가 달라도 같은 노드로 모이고, 이름이 같아도 다른 개체는 갈립니다.

</details>

**2.** 사전에 없는 이름을 그냥 버리지 않고 `:Candidate` 로 남기는 이유는?

<details><summary>정답 보기</summary>

사전에 없다는 것이 틀렸다는 뜻은 아니기 때문입니다. 사전은 2016년 시점이라 **그 뒤에 나온 약**이나 논문마다 다른 표기가 빠져 있습니다. 버리면 그 사실을 영영 잃고, 그대로 섞으면 기존 노드를 더럽힙니다. 따로 세워 두면 나중에 붙일 곳이 생겼을 때 승격할 수 있습니다.

</details>

## 2-2. 관계 시그니처: 주어 타입·목적어 타입·판정 기준

모델에게 "관계를 다 뽑아 줘"라고만 하면 같은 뜻인데 관계 이름이 제각각이 되고(치료한다·처방된다·쓰인다), 말이 안 되는 연결(유전자가 약을 치료한다)도 섞입니다. 그래서 **어떤 노드와 관계만 허용할지**를 미리 정하는 설계도가 필요합니다.

### 문법: 관계 시그니처
스키마가 정하는 두 가지 중 **관계 쪽**을 맡는 것이 **관계 시그니처**입니다. 관계마다 <strong>(주어 타입, 목적어 타입, 판정 기준)</strong>을 못 박은 한 줄입니다.

| 관계 | 주어 타입 | 목적어 타입 | 판정 기준 | 그래프에 있는 건수 |
|---|---|---|---|---|
| `TREATS` | Compound | Disease | 약이 질병을 치료한다. 질병의 원인이나 진행 자체에 작용한다 | 755 |
| `PALLIATES` | Compound | Disease | 약이 질병의 증상을 완화한다. 질병 자체는 그대로 두고 증상만 덜어 준다 | 390 |
| `BINDS` | Compound | Gene | 약이 그 유전자의 단백질에 결합한다. 그 유전자가 이 약의 대사·수송을 맡는다는 진술도 여기에 적는다. 그 대신 표적·효소·수송체는 가르지 않는다 | 11,571 |
| `UPREGULATES_CG` | Compound | Gene | 약이 그 유전자의 발현을 증가시킨다 | 18,756 |
| `DOWNREGULATES_CG` | Compound | Gene | 약이 그 유전자의 발현을 감소시킨다 | 21,102 |
| `ASSOCIATES` | Disease | Gene | 질병과 유전자 사이에 연관이 보고됐다 | 12,623 |
| `PRESENTS` | Disease | Symptom | 질병이 그 증상으로 나타난다 | 3,357 |
| `INCLUDES` | PharmacologicClass | Compound | 약효 분류가 그 약물을 포함한다. 그 약이 어느 계열에 속한다는 진술을 여기에 적는다 | 1,029 |

> `TREATS` 와 `PALLIATES` 는 주어도 목적어도 (Compound → Disease) 라 **타입으로는 갈리지 않습니다.** 둘을 가르는 것은 판정 기준 한 줄입니다. **치료**는 질병의 원인이나 진행 자체에 작용하는 것이고, **완화**는 질병을 그대로 둔 채 증상만 덜어 주는 것입니다.

<img src="images/ontology_schema.png" width="720">

### 핵심 판정 하나: TREATS 와 PALLIATES
두 관계는 **주어 타입도 목적어 타입도 같습니다**(Compound -> Disease). 그래서 타입만으로는 못 가릅니다. 갈라 주는 것은 **판정 기준**뿐입니다.

- **`TREATS`**: 약이 그 질병을 **치료**한다. 원인이나 진행 자체에 작용한다.
- **`PALLIATES`**: 약이 그 질병의 **증상을 완화**한다. 질병 자체는 그대로다.

이 구분이 장식이 아니라는 것은 건수를 보면 알 수 있습니다. 그래프에 `TREATS` 가 **755건**, `PALLIATES` 가 **390건** 입니다. 치료가 두 배도 안 됩니다. 우리가 아는 약의 상당수는 **병을 낫게 하는 게 아니라 증상을 다스립니다.** 둘을 한 관계로 뭉쳤다면 이 사실이 통째로 사라집니다.

논문 문장으로 보면 이렇습니다.

| 문장 | 판정 | 왜 |
|---|---|---|
| approved for the treatment of plaque psoriasis | `TREATS` | 그 병의 치료제로 승인 |
| used for the management of neuropathic pain | `PALLIATES` | 통증을 다스릴 뿐 |
| investigated as an oral treatment for multiple sclerosis | **판단 보류** | 연구되었다는 말이지 치료제라는 말이 아니다 |

세 번째 줄이 중요합니다. 논문은 "연구되었다"고만 했는데 추출기는 자주 `TREATS` 로 적어 옵니다. **"논문이 그렇게 보고했다"와 "효능이 입증됐다"는 다른 말입니다.** 이 구분을 트리플에 어떻게 남길지는 다음 시간에 다룹니다.

### 핵심 판정 둘: 대사·수송도 `BINDS` 로 적는다
우리가 읽을 논문은 대부분 **약물유전체** 논문이라 문장이 이런 꼴입니다.

> Four individuals received the antiplatelet agent clopidogrel and had a decreased activity CYP2C19 IM phenotype.

"CYP2C19 가 clopidogrel 을 활성화한다", 곧 **그 유전자가 이 약의 대사를 맡는다**는 말입니다. 그런데 우리 관계 8종에는 **대사라는 관계가 없습니다.** 이 사실을 어디에 적을까요?

우리가 쓰는 그래프는 이것을 **`BINDS`** 에 담아 두었습니다. 아래는 그래프에 실제로 들어 있는 관계입니다.

```text
Clopidogrel BINDS CYP2C19      Simvastatin BINDS SLCO1B1      Codeine  BINDS CYP2D6
Warfarin    BINDS CYP2C9       Irinotecan  BINDS UGT1A1       Warfarin BINDS VKORC1
```

약을 분해하는 **효소**(CYP2C19), 약을 실어 나르는 **수송체**(SLCO1B1), 약이 실제로 노리는 **표적**(VKORC1)이 전부 같은 `BINDS` 입니다. 기질이 효소의 활성부위에 붙는 것은 사실이니 화학적으로 틀린 표기는 아닙니다. 그래서 우리도 같은 판정을 씁니다.

> **"그 유전자가 이 약의 대사·수송을 맡는다"는 진술도 `BINDS` 로 적는다.**

**이 결정에는 대가가 있습니다.**

`BINDS` 하나에 **표적·효소·수송체의 구분이 사라집니다.** 그러면 이런 질문에 그래프가 답하지 못합니다.

- 이 약이 실제로 **노리는 표적**이 무엇인가
- 이 약을 **무엇이 분해하는가**

"그럼 `METABOLIZES` 를 하나 만들면 되지 않나" 싶을 텐데, 그러면 우리 그래프에 이미 `BINDS` 로 들어 있는 11,571건과 **같은 사실이 두 이름으로 갈립니다.** 새 관계가 없어서 문제가 아니라 **서로 다른 것이 문제**입니다.

여기서 이 단원의 가장 중요한 교훈이 나옵니다.

> **온톨로지가 구분하지 않기로 한 것은 나중에 되찾을 수 없습니다.**

트리플을 아무리 잘 뽑아도, 스키마가 두 가지를 한 이름으로 받기로 했으면 그 차이는 데이터에 남지 않습니다. 그래서 관계를 정하는 일이 추출보다 앞에 오고, 정할 때 **무엇을 포기하는지**까지 적어 두어야 합니다. 그래서 `BINDS` 의 판정 기준 마지막 줄이 **"그 대신 표적·효소·수송체는 가르지 않는다"** 입니다. 어디에 적을지만 적어 두면 코드를 읽는 사람은 무엇이 사라졌는지 알 수 없습니다.

(잃은 구분을 아주 조금 되찾는 방법이 하나 있습니다. 다음 시간 마지막 절에서 봅니다.)

In [ ]:
# 관계 규격을 모아 두는 한 곳. 관계를 더하거나 고칠 때는 여기만 손댄다
# 각 관계: (주어 타입, 목적어 타입, 판정 기준)
RELATION_SIGNATURES = {
    # 아래 둘은 주어·목적어 타입이 같다. 셋째 칸(판정 기준)만이 둘을 가른다
    "TREATS":           ("Compound", "Disease",
                         "약이 질병을 치료한다. "
                         "질병의 원인이나 진행 자체에 작용한다"),
    "PALLIATES":        ("Compound", "Disease",
                         "약이 질병의 증상을 완화한다. "
                         "질병 자체는 그대로 두고 증상만 덜어 준다"),
    # 대사·수송을 담을 관계는 따로 두지 않았다. 어느 관계에 적을지를 판정 기준에 적어 BINDS 로 모은다
    "BINDS":            ("Compound", "Gene",
                         "약이 그 유전자의 단백질에 결합한다. "
                         "그 유전자가 이 약의 대사·수송을 맡는다는 진술도 여기에 적는다. "
                         "그 대신 표적·효소·수송체는 가르지 않는다"),
    "UPREGULATES_CG":   ("Compound", "Gene", "약이 그 유전자의 발현을 증가시킨다"),
    "DOWNREGULATES_CG": ("Compound", "Gene", "약이 그 유전자의 발현을 감소시킨다"),
    "ASSOCIATES":       ("Disease", "Gene", "질병과 유전자 사이에 연관이 보고됐다"),
    "PRESENTS":         ("Disease", "Symptom", "질병이 그 증상으로 나타난다"),
    "INCLUDES":         ("PharmacologicClass", "Compound",
                         "약효 분류가 그 약물을 포함한다. "
                         "그 약이 어느 계열에 속한다는 진술을 여기에 적는다"),
}

# 노드 타입 5종. 지식그래프를 적재한 단원의 레이블과 글자까지 같아야 한다
# 타입 이름은 그래프의 노드 레이블 그대로다. 시그니처의 주어·목적어 칸이 이 중에서 나온다
NODE_TYPES = {"Compound", "Disease", "Gene", "Symptom", "PharmacologicClass"}

In [ ]:
subj_type, obj_type, criterion = RELATION_SIGNATURES["PALLIATES"]   # 값이 세 칸 튜플이라 한 줄에 풀어 받는다
print("PALLIATES 의 주어 타입:", subj_type)
print("PALLIATES 의 목적어 타입:", obj_type)
print("판정 기준:", criterion)
print("허용 관계 목록:", list(RELATION_SIGNATURES))

> 시그니처가 있으면 "이 관계는 약에서 유전자로만 간다"처럼 **엉뚱한 연결을 걸러낼 잣대**가 생깁니다. 실제로 걸러내는 검사는 다음 시간에 다룹니다. 오늘은 **설계**까지입니다.

### 🖐️ 함께 따라하기: 다른 도메인의 온톨로지를 세워 보기

**단위 프로젝트의 온톨로지를 직접 적어 봅니다.** 라이브러리 문서 그래프는 노드 타입도 관계도 의학 그래프와 전혀 다릅니다. 도메인이 바뀌면 온톨로지도 통째로 바뀐다는 것을 손으로 확인합니다.

`PYLIB_SIGNATURES` 라는 새 dict 에 아래 다섯 관계를 같은 세 칸 꼴로 담으세요. `RELATION_SIGNATURES` 는 건드리지 않습니다(아래 절에서 계속 씁니다).

| 관계 | 주어 타입 | 목적어 타입 | 판정 기준 |
|---|---|---|---|
| `DEMONSTRATES` | Document | ApiElement | 문서가 그 API 를 예제로 보여 준다 |
| `RAISES` | ApiElement | ApiElement | 그 API 가 이 예외를 낸다 |
| `INCLUDES` | Document | Change | 릴리스 문서가 그 변경을 담고 있다 |
| `AFFECTS` | Change | ApiElement | 그 변경이 이 API 를 바꾼다 |
| `REFERENCES` | Change | Issue | 그 변경이 이 이슈를 가리킨다 |

담은 뒤 허용 관계 목록과, `AFFECTS` 의 세 칸을 풀어 출력하세요.

**확인 기준**: 목록이 다섯 개이고 `AFFECTS` 의 주어 타입이 `Change`, 목적어 타입이 `ApiElement` 입니다. **주어가 문서가 아니라 변경**이라는 점을 눈여겨보세요.

In [ ]:
# 🖐️ 함께 따라하기 (아래 순서대로 직접 작성해 보세요)
# 1) PYLIB_SIGNATURES 에 위 표의 다섯 관계를 (주어 타입, 목적어 타입, 판정 기준) 으로 담는다
# 2) list(PYLIB_SIGNATURES) 로 허용 관계 목록을 출력한다
# 3) AFFECTS 의 세 칸을 풀어 받아 각각 출력한다

### 🖐️ 함께 따라하기: 라이브러리 트리플을 Cypher 스크립트로

**단위 프로젝트 데이터로 마무리합니다.** `data/pylibs_triples.jsonl` 에 라이브러리 문서에서 뽑은 트리플 아홉 건이 들어 있습니다. 칸 이름은 교안과 같습니다(`subject`·`subject_type`·`relation`·`object`·`object_type`·`evidence`).

다섯 칸 트리플의 리스트를 받아 각 트리플을 `triple_to_cypher` 로 옮긴 뒤 빈 줄로 이어 붙인 **하나의 Cypher 스크립트 문자열**을 돌려주는 `triples_to_script(triples)` 를 완성하세요. 아래 `sample_triples` 로 실행해 결과를 출력합니다.

**확인 기준**: `MERGE` 아홉 줄이 세 덩어리로 나뉘어 찍힙니다(트리플 세 개 x 세 줄). 덩어리 사이에는 빈 줄이 하나씩 있습니다. 노드 열쇠 자리에 **이름이 그대로** 들어가 있는 것도 확인하세요. 이 데이터는 아직 id 로 바꾸지 않았습니다. 단위 프로젝트에서 그 해소를 직접 하게 됩니다.

In [ ]:
# 단위 프로젝트 데이터에서 앞 세 건만 읽어 온다. 칸 이름이 교안과 같아 그대로 넘길 수 있다
pylib_rows = [json.loads(line) for line
              in Path("data/pylibs_triples.jsonl").read_text(encoding="utf-8").splitlines()]
sample_triples = [(r["subject"], r["subject_type"], r["relation"], r["object"], r["object_type"])
                  for r in pylib_rows[:3]]
for tp in sample_triples:
    print(tp)

In [ ]:
# 🖐️ 함께 따라하기 (아래 순서대로 직접 작성해 보세요)
# 1) triples_to_script(triples) 를 정의한다
# 2) 각 트리플을 triple_to_cypher 로 옮겨 리스트에 모은다
# 3) 빈 줄('\n\n')로 이어 붙인 하나의 문자열을 return 한다
# 4) sample_triples 로 호출해 출력한다

### ✅ 바로 확인 퀴즈

**1.** 추출기가 트리플 **(CYP2C19, BINDS, Clopidogrel)** 을 뽑아 왔습니다. 관계 시그니처로 볼 때 무엇이 문제인가요?

<details><summary>정답 보기</summary>

`BINDS` 의 **주어 타입은 Compound** 인데 이 트리플의 주어는 <strong>Gene(CYP2C19)</strong>입니다. 주어와 목적어가 뒤바뀐 트리플이라 방향이 어긋납니다. 그대로 넣으면 "유전자가 약에 결합한다"가 되어, "이 약이 어떤 유전자에 붙는가"를 묻는 쿼리에 잡히지 않습니다.

</details>

**2.** `TREATS` 와 `PALLIATES` 는 주어 타입도 목적어 타입도 같습니다. 그런데도 두 관계를 나눠 둔 이유는 무엇일까요?

<details><summary>정답 보기</summary>

질문이 다르기 때문입니다. "이 병을 치료하는 약"과 "이 병의 증상을 완화하는 약"은 임상에서 전혀 다른 답이고, 하나로 뭉치면 그 질문에 답할 수 없습니다. 타입으로는 못 가르므로 **판정 기준을 시그니처에 적어** 사람과 모델이 같은 기준으로 고르게 합니다.

</details>

**3.** 반대로 대사·수송은 표적과 **한 관계로 뭉쳤습니다**(`BINDS`). 2번과 정반대 결정인데, 그래도 되는 이유는 무엇이고 무엇을 잃었나요?

<details><summary>정답 보기</summary>

그래도 되는 이유는 **우리가 얹을 그래프가 이미 그렇게 담고 있기** 때문입니다. 여기서 `METABOLIZES` 를 새로 만들면 같은 사실이 두 이름으로 갈려, 뒤에서 두 데이터를 합칠 때 서로 못 알아봅니다. **없는 것이 문제가 아니라 서로 다른 것이 문제**입니다.

잃은 것은 "이 약의 표적이 무엇인가"와 "이 약을 무엇이 분해하는가"를 **그래프에 물을 수 없게 된 것**입니다. 그리고 이건 나중에 되찾을 수 없습니다. 스키마가 안 가르기로 한 차이는 데이터에 남지 않기 때문입니다.

</details>

---
## 이번 강의 정리

| 개념 | 핵심 |
|---|---|
| 트리플 | 사실 하나 = (주어, 관계, 목적어). 지식의 최소 단위 |
| LPG 매핑 | 주어·목적어 → 노드, 관계 → 대문자 스네이크 엣지. `MERGE` 로 멱등 적재 |
| 노드의 키 | **이름이 아니라 id**. 이름이 겹치는 개체가 실제로 9건 |
| `:Candidate` | 사전에 없거나 애매한 이름은 격리. 버리지도 섞지도 않는다 |
| 관계 시그니처 | 관계마다 (주어 타입, 목적어 타입, 판정 기준). 오늘은 8종 |
| TREATS / PALLIATES | 타입이 같아 **판정 기준**으로만 갈린다 |
| BINDS | 대사·수송도 여기 담는다. 표적과의 구분이 사라지는 것이 그 대가 |

- 논문 한 문장은 여러 트리플로 쪼개지고, 문단의 주인공을 주어로 모으면 사실이 쌓입니다.
- 트리플의 세 칸이 그래프의 노드-관계-노드로 **1:1 매핑**됩니다.
- 시그니처는 **엉뚱한 연결을 걸러낼 잣대**입니다(실제 검사는 다음 시간).
- **온톨로지가 구분하지 않기로 한 것은 나중에 되찾을 수 없습니다.** 관계를 정할 때 무엇을 포기하는지까지 적어 둡니다.

## ⏭️ 예고: 다음 시간

오늘 만든 시그니처들을 코드 한 곳의 <strong>온톨로지(단일 계약)</strong>로 모읍니다. 그리고 **구조화 출력**으로 논문에서 트리플을 뽑고(근거 evidence 포함), **온톨로지를 주입한 추출 프롬프트**를 설계한 뒤, 뽑은 트리플을 **Neo4j 에 id 로 MERGE** 하고 사전에 없는 이름은 `:Candidate` 로 격리합니다.

중간에 두 가지를 더 봅니다. 모델이 관계를 헷갈릴 때 쓰는 **CoT**(생각을 단계로 쪼개기), 그리고 "논문이 보고했다"와 "정리된 데이터베이스가 그렇다"를 그래프에 남기는 **근거 등급**입니다.

수고하셨습니다!